# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [11]:
%pip install -U huggingface_hub polars duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 1.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 1.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [huggingface_hub] [huggingface_hub]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "IndonesiaAI/cleaned-data-split-0"

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

parquet_files = [f for f in files if f.endswith(".parquet")]
print("File Parquet yang tersedia di repositori:")
for f in parquet_files:
    print("-", f)

File Parquet yang tersedia di repositori:
- data/train-00000-of-00006-042e4a2154d9270b.parquet
- data/train-00001-of-00006-8666365fa80b1ec5.parquet
- data/train-00002-of-00006-8ac9b553e82a7f8f.parquet
- data/train-00003-of-00006-25183230856f688a.parquet
- data/train-00004-of-00006-a7a9cb4b60d46e79.parquet
- data/train-00005-of-00006-a7fd9cf190643493.parquet


In [1]:
import os

# Kurangi masalah pada progres interaktif dan jalur unduhan Xet
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

PROJECT_ROOT = Path("/home/jovyan/work")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "IndonesiaAI/cleaned-data-split-0"

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
parquet_files = sorted(f for f in files if f.endswith(".parquet"))

print("Jumlah file Parquet yang ditemukan:", len(parquet_files))
for f in parquet_files:
    print(f)

Jumlah file Parquet yang ditemukan: 6
data/train-00000-of-00006-042e4a2154d9270b.parquet
data/train-00001-of-00006-8666365fa80b1ec5.parquet
data/train-00002-of-00006-8ac9b553e82a7f8f.parquet
data/train-00003-of-00006-25183230856f688a.parquet
data/train-00004-of-00006-a7a9cb4b60d46e79.parquet
data/train-00005-of-00006-a7fd9cf190643493.parquet


In [2]:
for i, filename in enumerate(parquet_files, start=1):
    print(f"[{i}/{len(parquet_files)}] Mengunduh: {filename}")

    downloaded_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        local_dir=RAW_DIR,
    )

    print("Selesai:", downloaded_path)

[1/6] Mengunduh: data/train-00000-of-00006-042e4a2154d9270b.parquet


Selesai: /home/jovyan/work/data/raw/data/train-00000-of-00006-042e4a2154d9270b.parquet
[2/6] Mengunduh: data/train-00001-of-00006-8666365fa80b1ec5.parquet
Selesai: /home/jovyan/work/data/raw/data/train-00001-of-00006-8666365fa80b1ec5.parquet
[3/6] Mengunduh: data/train-00002-of-00006-8ac9b553e82a7f8f.parquet
Selesai: /home/jovyan/work/data/raw/data/train-00002-of-00006-8ac9b553e82a7f8f.parquet
[4/6] Mengunduh: data/train-00003-of-00006-25183230856f688a.parquet
Selesai: /home/jovyan/work/data/raw/data/train-00003-of-00006-25183230856f688a.parquet
[5/6] Mengunduh: data/train-00004-of-00006-a7a9cb4b60d46e79.parquet
Selesai: /home/jovyan/work/data/raw/data/train-00004-of-00006-a7a9cb4b60d46e79.parquet
[6/6] Mengunduh: data/train-00005-of-00006-a7fd9cf190643493.parquet
Selesai: /home/jovyan/work/data/raw/data/train-00005-of-00006-a7fd9cf190643493.parquet


In [3]:
parquet_files_local = sorted(RAW_DIR.rglob("*.parquet"))

print("Jumlah file lokal:", len(parquet_files_local))

for f in parquet_files_local:
    print(f"{f.name}: {f.stat().st_size / 1_000_000:.2f} MB")

Jumlah file lokal: 6
train-00000-of-00006-042e4a2154d9270b.parquet: 153.50 MB
train-00001-of-00006-8666365fa80b1ec5.parquet: 153.29 MB
train-00002-of-00006-8ac9b553e82a7f8f.parquet: 151.92 MB
train-00003-of-00006-25183230856f688a.parquet: 152.17 MB
train-00004-of-00006-a7a9cb4b60d46e79.parquet: 152.67 MB
train-00005-of-00006-a7fd9cf190643493.parquet: 152.77 MB


In [6]:
from pathlib import Path
import duckdb
import polars as pl

# Tentukan lokasi proyek secara langsung
PROJECT_ROOT = Path("/home/jovyan/work")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Cari file Parquet di folder raw dan seluruh subfoldernya
PARQUET_FILES = sorted(RAW_DIR.rglob("*.parquet"))

print("Lokasi folder raw:", RAW_DIR)
print("Folder tersedia:", RAW_DIR.exists())
print("Jumlah file Parquet:", len(PARQUET_FILES))

if not PARQUET_FILES:
    raise FileNotFoundError(
        f"File Parquet belum ditemukan di {RAW_DIR}. "
        "Pastikan proses unduhan sudah selesai."
    )

for file in PARQUET_FILES:
    print(f"{file.name} | {file.stat().st_size / 1_000_000:.2f} MB")

# Membaca semua file Parquet
df = pl.scan_parquet([str(file) for file in PARQUET_FILES])

print("\nSkema dataset:")
print(df.collect_schema())

print("\nLima baris pertama:")
display(df.head(5).collect())

Lokasi folder raw: /home/jovyan/work/data/raw
Folder tersedia: True
Jumlah file Parquet: 6
train-00000-of-00006-042e4a2154d9270b.parquet | 153.50 MB
train-00001-of-00006-8666365fa80b1ec5.parquet | 153.29 MB
train-00002-of-00006-8ac9b553e82a7f8f.parquet | 151.92 MB
train-00003-of-00006-25183230856f688a.parquet | 152.17 MB
train-00004-of-00006-a7a9cb4b60d46e79.parquet | 152.67 MB
train-00005-of-00006-a7fd9cf190643493.parquet | 152.77 MB

Skema dataset:
Schema({'qid': Int64, 'question': String, 'date': String, 'metadata': List(String), 'response_j': String, 'response_k': String})

Lima baris pertama:


qid,question,date,metadata,response_j,response_k
i64,str,str,list[str],str,str
26351,"""I was studying for icing and a…","""2016/03/23""","[""https://aviation.stackexchange.com/questions/26351"", ""https://aviation.stackexchange.com"", ""https://aviation.stackexchange.com/users/6831/""]","""Lets start with the very basic…","""For those recommended actions …"
26351,"""I was studying for icing and a…","""2016/03/23""","[""https://aviation.stackexchange.com/questions/26351"", ""https://aviation.stackexchange.com"", ""https://aviation.stackexchange.com/users/6831/""]","""Lets start with the very basic…","""I have a theory on applying th…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""OK, hopefully supplying a *use…","""If you want something where yo…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""If you're on Windows, what abo…","""If you want something where yo…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""OK, hopefully supplying a *use…","""If you're on Windows, what abo…"


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [8]:
from pathlib import Path
import polars as pl

# Lokasi dataset yang benar
RAW_DIR = Path("/home/jovyan/work/data/raw")

# Cari file Parquet di dalam folder dan subfolder
PARQUET_FILES = sorted(RAW_DIR.rglob("*.parquet"))

print("Folder dataset:", RAW_DIR)
print("Folder tersedia:", RAW_DIR.exists())
print("Jumlah file:", len(PARQUET_FILES))

if not PARQUET_FILES:
    raise FileNotFoundError(
        "Belum ada file Parquet. Periksa proses unduhan dataset."
    )

# Baca seluruh file Parquet sebagai satu dataset
lazy_data = pl.scan_parquet([str(file) for file in PARQUET_FILES])

print("\nSkema dataset:")
print(lazy_data.collect_schema())

print("\nLima baris pertama:")
display(lazy_data.head(5).collect())

Folder dataset: /home/jovyan/work/data/raw
Folder tersedia: True
Jumlah file: 6

Skema dataset:
Schema({'qid': Int64, 'question': String, 'date': String, 'metadata': List(String), 'response_j': String, 'response_k': String})

Lima baris pertama:


qid,question,date,metadata,response_j,response_k
i64,str,str,list[str],str,str
26351,"""I was studying for icing and a…","""2016/03/23""","[""https://aviation.stackexchange.com/questions/26351"", ""https://aviation.stackexchange.com"", ""https://aviation.stackexchange.com/users/6831/""]","""Lets start with the very basic…","""For those recommended actions …"
26351,"""I was studying for icing and a…","""2016/03/23""","[""https://aviation.stackexchange.com/questions/26351"", ""https://aviation.stackexchange.com"", ""https://aviation.stackexchange.com/users/6831/""]","""Lets start with the very basic…","""I have a theory on applying th…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""OK, hopefully supplying a *use…","""If you want something where yo…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""If you're on Windows, what abo…","""If you want something where yo…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""OK, hopefully supplying a *use…","""If you're on Windows, what abo…"


In [9]:
contoh_data = lazy_data.head(10).collect()
contoh_data

qid,question,date,metadata,response_j,response_k
i64,str,str,list[str],str,str
26351,"""I was studying for icing and a…","""2016/03/23""","[""https://aviation.stackexchange.com/questions/26351"", ""https://aviation.stackexchange.com"", ""https://aviation.stackexchange.com/users/6831/""]","""Lets start with the very basic…","""For those recommended actions …"
26351,"""I was studying for icing and a…","""2016/03/23""","[""https://aviation.stackexchange.com/questions/26351"", ""https://aviation.stackexchange.com"", ""https://aviation.stackexchange.com/users/6831/""]","""Lets start with the very basic…","""I have a theory on applying th…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""OK, hopefully supplying a *use…","""If you want something where yo…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""If you're on Windows, what abo…","""If you want something where yo…"
77133,"""There seems to be a lot of sof…","""2009/11/30""","[""https://superuser.com/questions/77133"", ""https://superuser.com"", ""https://superuser.com/users/19668/""]","""OK, hopefully supplying a *use…","""If you're on Windows, what abo…"
8499,"""I am writing an application, r…","""2013/05/30""","[""https://crypto.stackexchange.com/questions/8499"", ""https://crypto.stackexchange.com"", ""https://crypto.stackexchange.com/users/7065/""]","""If a user has a copy of both t…","""Your master secret is **never*…"
2055713,"""In his book ""Men of Mathematic…","""2016/12/12""","[""https://math.stackexchange.com/questions/2055713"", ""https://math.stackexchange.com"", ""https://math.stackexchange.com/users/-1/""]","""1. Feller's first volume for P…","""It's a bit more advanced than …"
2055713,"""In his book ""Men of Mathematic…","""2016/12/12""","[""https://math.stackexchange.com/questions/2055713"", ""https://math.stackexchange.com"", ""https://math.stackexchange.com/users/-1/""]","""I don’t agree with Bell on thi…","""It's a bit more advanced than …"
2055713,"""In his book ""Men of Mathematic…","""2016/12/12""","[""https://math.stackexchange.com/questions/2055713"", ""https://math.stackexchange.com"", ""https://math.stackexchange.com/users/-1/""]","""1. Feller's first volume for P…","""In the early '70s, I used two …"


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [11]:
from pathlib import Path
import duckdb

# Lokasi folder dataset
RAW_DIR = Path("/home/jovyan/work/data/raw")

# Cari seluruh file Parquet
parquet_files = sorted(RAW_DIR.rglob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"Tidak ditemukan file Parquet di {RAW_DIR}. "
        "Pastikan unduhan dataset sudah selesai."
    )

print(f"Jumlah file ditemukan: {len(parquet_files)}")

# Hubungkan ke DuckDB
connection = duckdb.connect()

# Baca file Parquet langsung dari disk
profil_duckdb = connection.execute(
    "SUMMARIZE SELECT * FROM read_parquet(?)",
    [[str(file) for file in parquet_files]]
).pl()

display(profil_duckdb)

Jumlah file ditemukan: 6


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""qid""","""BIGINT""","""1""","""74661602""",264619,"""2798925.6786795994""","""9036720.753639778""","""26941""","""105289""","""457156""",1050257,0.00
"""question""","""VARCHAR""","""!!! I found an interesting sol…","""﻿ What would a political syste…",374355,null,null,null,null,null,1050257,0.00
"""date""","""VARCHAR""","""2008/08/01""","""2023/03/04""",4030,null,null,null,null,null,1050257,0.00
"""metadata""","""VARCHAR[]""","""['https://3dprinting.meta.stac…","""['https://writers.stackexchang…",374242,null,null,null,null,null,1050257,0.00
"""response_j""","""VARCHAR""","""""","""ﷲ = الله ====""",595801,null,null,null,null,null,1050257,0.00
"""response_k""","""VARCHAR""","""! Just as well that I have a…","""Watch only produces notificat…",631183,null,null,null,null,null,1050257,0.00


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

## Catatan Milestone 1

### 1. Ukuran Data
Dataset yang digunakan adalah **IndonesiaAI - cleaned-data-split-0** dari platform Hugging Face Datasets, dengan split `train` dan format Parquet. Dataset terdiri dari 6 file Parquet yang disimpan di direktori `data/raw/`. Berdasarkan informasi dataset, ukuran penyimpanan lokalnya sekitar **874 MB**.

### 2. Jumlah Baris
Dataset memiliki sekitar **1.050.257 baris**. Jumlah tersebut telah memenuhi persyaratan tugas Analisis Big Data, yaitu lebih dari 1 juta baris.

### 3. Tipe Kolom
Dataset berisi data teks dan metadata. Berdasarkan contoh struktur dataset, kolom yang tersedia meliputi `qid`, `question`, `date`, `metadata`, `response_j`, dan `response_k`. Tipe data aktual setiap kolom akan dikonfirmasi melalui profiling menggunakan Polars dan DuckDB.

### 4. Kolom dengan Null Terbanyak
Jumlah nilai null pada setiap kolom belum ditetapkan karena hasil profiling aktual masih perlu diperiksa. Pemeriksaan dilakukan untuk mengetahui kolom yang memiliki nilai kosong terbanyak sehingga dapat ditentukan prioritas penanganannya pada Milestone 2.

### 5. Anomali yang Perlu Ditangani pada Milestone 2
Beberapa aspek yang perlu diperiksa lebih lanjut meliputi:
- Nilai null atau data kosong pada kolom teks dan metadata.
- Baris duplikat yang berpotensi memengaruhi hasil analisis.
- Konsistensi format tanggal pada kolom `date`.
- Ketidaksesuaian tipe data atau format isi kolom.
- Konsistensi isi teks pada kolom `question`, `response_j`, dan `response_k`.

Temuan tersebut merupakan aspek pemeriksaan yang direncanakan, bukan anomali yang sudah dipastikan ditemukan.

### 6. Rencana Milestone 2
Pada Milestone 2, data akan diprofilkan dan dibersihkan menggunakan Polars, sedangkan DuckDB digunakan untuk menghasilkan ringkasan statistik setiap kolom melalui perintah `SUMMARIZE`. Penanganan nilai kosong, duplikat, dan format tanggal akan dilakukan berdasarkan hasil pemeriksaan agar data tetap sesuai dengan tujuan analisis.

### Kesimpulan
Dataset IndonesiaAI - cleaned-data-split-0 dipilih karena memiliki lebih dari satu juta baris dan tersedia dalam format Parquet. Dataset ini akan digunakan untuk mempelajari karakteristik data, mengevaluasi kualitas data, dan melakukan proses pembersihan sebelum analisis lanjutan.
